<a href="https://colab.research.google.com/github/levina-ai/regulatory-chatbot/blob/main/priips_fca_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 PRIIPS/KID Regulatory Comparison Chatbot

A **Retrieval-Augmented Generation (RAG)** chatbot that compares **EU PRIIPS KID** vs **UK FCA** regulatory frameworks.

**Stack:** Azure OpenAI · Azure AI Search · LangChain

**Author:** Anastasiya Levina

---

### Architecture
```
PDF Documents (EU PRIIPS + UK FCA)
        ↓
Chunking (RecursiveCharacterTextSplitter)
        ↓
Embeddings (text-embedding-ada-002)
        ↓
Vector Store (Azure AI Search)
        ↓
Retrieval → GPT-4o-mini → Answer
```



In [2]:
!pip install langchain langchain-openai langchain-community langchain-text-splitters \
             azure-search-documents azure-identity \
             openai pypdf python-dotenv -q

In [41]:
import os

os.environ['AZURE_OPENAI_ENDPOINT'] = 'YOUR_AZURE_OPENAI_ENDPOINT'  # https://xxx.openai.azure.com/
os.environ['AZURE_OPENAI_API_KEY']  = 'YOUR_AZURE_OPENAI_API_KEY'
os.environ['AZURE_SEARCH_ENDPOINT'] = 'YOUR_AZURE_SEARCH_ENDPOINT'  # https://xxx.search.windows.net
os.environ['AZURE_SEARCH_API_KEY']  = 'YOUR_AZURE_SEARCH_API_KEY'
os.environ['AZURE_SEARCH_INDEX']    = 'priips-index'

print('✅ Credentials loaded')

## Upload Regulatory Documents

You will be prompted to upload two PDFs:

1. EU PRIIPS — Commission Delegated Regulation (EU) 2017/653
https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:02017R0653-20230101

2. UK FCA — Policy Statement
https://www.fca.org.uk/publication/policy/ps25-20.pdf


In [42]:
from google.colab import files

print('⬆️  Upload EU PRIIPS document')
eu_upload   = files.upload()
eu_filename = list(eu_upload.keys())[0]

print('⬆️  Upload UK FCA document')
uk_upload   = files.upload()
uk_filename = list(uk_upload.keys())[0]

print('✅ Both documents uploaded')

⬆️  Upload EU PRIIPS document


Saving CELEX_02017R0653-20230101_EN_TXT.pdf to CELEX_02017R0653-20230101_EN_TXT (3).pdf
⬆️  Upload UK FCA document


Saving ps25-20.pdf to ps25-20 (2).pdf
✅ Both documents uploaded


## Load and Chunk Documents

KEY CONCEPTS:


* Chunking — splits large documents into smaller overlapping       
pieces so the model can fit relevant text into its context window.
* Chunk_size — max characters per chunk (1000 is a good default for dense regulatory text).
* Chunk_overlap — characters shared between adjacent chunks, preventing answers from being split at boundaries.
* Source_label — metadata tag so we always know whether a chunk     came from the EU or UK document.


In [43]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_and_chunk(filename, source_label):
    """Load a PDF and split into overlapping chunks with a source tag."""
    loader    = PyPDFLoader(filename)
    documents = loader.load()

    for doc in documents:
        doc.metadata['source_label'] = source_label

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = splitter.split_documents(documents)
    print(f'  {source_label}: {len(chunks)} chunks from {len(documents)} pages')
    return chunks

print('Loading and chunking documents...')
eu_chunks  = load_and_chunk(eu_filename, 'EU_PRIIPS')
uk_chunks  = load_and_chunk(uk_filename, 'UK_FCA')
all_chunks = eu_chunks + uk_chunks

print(f'\n✅ Total chunks ready for indexing: {len(all_chunks)}')

Loading and chunking documents...
  EU_PRIIPS: 280 chunks from 85 pages
  UK_FCA: 495 chunks from 194 pages

✅ Total chunks ready for indexing: 775


## Embed and Index in Azure AI Search

KEY CONCEPTS:

* Embeddings — numerical vector representations of text.Semantically similar text produces similar vectors, enabling meaning-based search (not just keywords).
* text-embedding-ada-002 — Azure OpenAI model that converts text into 1536-dimensional vectors.
* Vector store — database optimised for storing and searching vectors. Azure AI Search is our vector store.
* Indexing — embedding each chunk and storin

In [49]:
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import AzureSearch

print('Initialising embedding model (text-embedding-ada-002)...')
embeddings = AzureOpenAIEmbeddings(
    azure_deployment='text-embedding-ada-002',
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    api_key=os.environ['AZURE_OPENAI_API_KEY'],
    api_version='2024-02-01'
)

print('Connecting to Azure AI Search...')
vector_store = AzureSearch(
    azure_search_endpoint=os.environ['AZURE_SEARCH_ENDPOINT'],
    azure_search_key=os.environ['AZURE_SEARCH_API_KEY'],
    index_name=os.environ['AZURE_SEARCH_INDEX'],
    embedding_function=embeddings.embed_query
)

print(f'Indexing {len(all_chunks)} chunks — please wait 3–5 minutes...')
vector_store.add_documents(all_chunks)
print('✅ Documents indexed successfully in Azure AI Search')

Initialising embedding model (text-embedding-ada-002)...
Connecting to Azure AI Search...
Indexing 775 chunks — please wait 3–5 minutes...
✅ Documents indexed successfully in Azure AI Search


## Build the RAG Pipeline

KEY CONCEPTS:

RAG (Retrieval-Augmented Generation) — instead of relying on       the LLM's training data, we retrieve relevant document
chunks and augment the prompt with that context, grounding answers in the actual source documents.

Pipeline:
1. User asks a question
2. Question → vector (same embedding model)
3. Azure AI Search finds k most similar chunks
4. Chunks + question → GPT-4o-mini as context
5. GPT-4o-mini generates a grounded answer

temperature=0 — deterministic output, if you ask the same question twice you need the same answer.

similarity_search — called directly to avoid a known version  conflict between LangChain and Azure AI

Search's hybrid search interface.

In [50]:
from langchain_openai import AzureChatOpenAI

print('Initialising GPT-4o-mini...')
llm = AzureChatOpenAI(
    azure_deployment='gpt-4o-mini',
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    api_key=os.environ['AZURE_OPENAI_API_KEY'],
    api_version='2024-02-01',
    temperature=0
)

SYSTEM_PROMPT = """You are a regulatory compliance assistant specialising in
EU PRIIPS KID and UK FCA regulations.
Use ONLY the provided context to answer. Do not use prior knowledge.
When comparing frameworks, clearly label EU vs UK differences. Make the answers conscise and in plain-english.
If the answer is not present in the context, say so explicitly."""


Initialising GPT-4o-mini...


In [53]:
def ask(question):
    """
    RAG pipeline:
      1. Embed question → retrieve 3 most relevant chunks from Azure AI Search
      2. Build grounded prompt from retrieved context
      3. Send to GPT-4o-mini and display answer with source attribution
    """
    # Step 1 — Retrieve
    docs    = vector_store.similarity_search(query=question, k=3)
    context = '\n\n'.join([d.page_content for d in docs])

    # Step 2 — Build prompt
    prompt  = f'{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:'

    # Step 3 — Generate
    response = llm.invoke(prompt)

    # Step 4 — Display
    sources = list(set([d.metadata.get('source_label', 'unknown') for d in docs]))
    pages   = [f"p.{d.metadata.get('page', '?')}" for d in docs]

    print(f"\n{'='*60}")
    print(f'🔍  {question}')
    print(f"{'='*60}")
    print(f'\n{response.content}')
    print(f'\n📄  Sources: {sources} | Pages: {pages}')

print('✅ RAG pipeline ready')

✅ RAG pipeline ready


## Run Queries

In [54]:
# Single framework questions
ask('What are the cost disclosure requirements under EU PRIIPS?')


🔍  What are the cost disclosure requirements under EU PRIIPS?

Under EU PRIIPs, the cost disclosure requirements are as follows:

1. **Costs Over Time Table**: PRIIP manufacturers must provide a summary cost indicator that shows the total aggregated costs of the PRIIP as a single number in both monetary and percentage terms for different time periods specified in Annex VI.

2. **Prominent Warning**: A prominent warning must be included, where applicable, about any additional costs that may be charged by persons advising on or selling the PRIIP.

3. **Composition of Costs Table**: Manufacturers must specify summary indicators of various types of costs in this section of the key information document. 

These requirements ensure that investors are clearly informed about the costs associated with the PRIIPs they are considering.

📄  Sources: ['EU_PRIIPS'] | Pages: ['p.7', 'p.7', 'p.7']


In [37]:
ask('What are the cost disclosure requirements under UK FCA?')


🔍  What are the cost disclosure requirements under UK FCA?

The context does not provide information on the cost disclosure requirements under UK FCA.

📄  Sources: [] | Pages: []


In [55]:
ask('How do performance scenarios differ between the EU and UK regimes?')


🔍  How do performance scenarios differ between the EU and UK regimes?

The context does not provide specific details on how performance scenarios differ between the EU and UK regimes. Therefore, I cannot answer this question explicitly.

📄  Sources: ['UK_FCA'] | Pages: ['p.54', 'p.54', 'p.43']


In [56]:
ask('What are the differences in risk measures for structured products?')


🔍  What are the differences in risk measures for structured products?

**EU vs UK Differences in Risk Measures for Structured Products:**

**EU:**
- Structured products are categorized into capital guaranteed notes, structured capital-at-risk products (SCARPs), and structured deposits.
- Manufacturers must calculate the risk score based on the volatility of the underlying assets or asset classes.
- A bespoke approach for measuring risk and return is suggested due to the unique features of structured products.
- The Value-at-Risk Equivalent Volatility (VEV) methodology is required for calculating volatility.

**UK:**
- Similar categorization of structured products is acknowledged.
- Manufacturers are also required to calculate risk scores based on volatility but must adjust for additional risks.
- There is an emphasis on accurately reflecting the risk of products, particularly regarding leverage thresholds.
- The use of VEV methodology is recognized, aligning with the EU approach.

Ove

In [57]:
print('Interactive mode — type your question or exit to quit\n')
while True:
    question = input('Your question: ').strip()
    if question.lower() == 'exit':
        print('Exiting.')
        break
    if question:
        ask(question)

Interactive mode — type your question or exit to quit

Your question: How should performance metrics be displayed?

🔍  How should performance metrics be displayed?

**EU vs UK Differences:**

**EU:** For CCIs with past performance, manufacturers must produce a line graph showing the product’s past 10 years of performance, plotting pound value on the y-axis and calendar years on the x-axis. Performance should be presented over rolling 12-month periods with a minimum of quarterly data points and an initial investment amount of £10,000, which can be substituted by the distributor with the investor’s actual amount. If products have between 3 months and 10 years of performance data, they must present the available data.

**UK:** The context does not provide specific UK regulations regarding the display of performance metrics, so no comparison can be made. 

In summary, performance metrics should be displayed in a line graph format, showing 10 years of performance, with specific requirements